# Hands-on 35: Studi Kasus A - Smart Room Monitoring Lengkap

Buku Referensi: *IoT Terintegrasi dengan Machine Learning & Data Analytics* (Onno W. Purbo), Bab 14.

Sistem lengkap: sensor/simulator -> MQTT -> subscriber Python (validasi, fitur
lag/rolling) -> klasifikasi nyaman/tidak_nyaman -> storage. Label mengikuti aturan buku:
nyaman jika suhu 24-28 C, kelembaban 45-65%, CO2 < 1000.

In [1]:
import os
from pathlib import Path

while not (Path.cwd() / "scripts" / "mqtt").is_dir():
    os.chdir("..")
print("[INFO] Working directory:", Path.cwd())

[INFO] Working directory: /home/sandbox/iot-ml-book


## 1. Latih Model (baseline LR vs Random Forest)

In [2]:
import subprocess, sys
r = subprocess.run([sys.executable, "scripts/case_a_train_room_model.py"], capture_output=True, text=True)
print(r.stdout)

[INFO] Distribusi label: {'tidak_nyaman': 1833, 'nyaman': 325}

=== Baseline Logistic Regression (accuracy): 0.9074 ===

=== Random Forest - Confusion Matrix ===
[[ 64   1]
 [  0 367]]

=== Random Forest - Classification Report ===
              precision    recall  f1-score   support

      nyaman       1.00      0.98      0.99        65
tidak_nyaman       1.00      1.00      1.00       367

    accuracy                           1.00       432
   macro avg       1.00      0.99      1.00       432
weighted avg       1.00      1.00      1.00       432

[SUKSES] Model disimpan ke: models/case_a_room_rf.joblib
[SUKSES] Laporan disimpan ke: outputs/reports/case_a_evaluation.csv



## 2. Deployment Mini (broker + subscriber + publisher)

In [3]:
import time

broker = subprocess.Popen([sys.executable, "scripts/mqtt/run_broker.py"],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)
try:
    sub = subprocess.Popen(
        [sys.executable, "scripts/case_a_room_subscriber.py",
         "--max-messages", "8", "--timeout", "30"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    time.sleep(1)
    subprocess.run([sys.executable, "scripts/case_a_room_publisher.py",
                    "--count", "8", "--interval", "0.3"],
                   capture_output=True, text=True)
    out, _ = sub.communicate()
    print(out)
finally:
    broker.terminate()

[INFO] Subscribe ke casea/room01/telemetry
[2026-10-22T08:11:42] -> status=nyaman
[2026-10-22T08:11:42] -> status=tidak_nyaman
[2026-10-22T08:11:43] -> status=nyaman
[2026-10-22T08:11:43] -> status=nyaman
[2026-10-22T08:11:43] -> status=tidak_nyaman
[2026-10-22T08:11:44] -> status=nyaman
[2026-10-22T08:11:44] -> status=tidak_nyaman
[2026-10-22T08:11:44] -> status=nyaman
[SUKSES] 8 prediksi disimpan ke outputs/reports/case_a_room_predictions.csv



## 3. Verifikasi Hasil

In [4]:
import pandas as pd
df = pd.read_csv("outputs/reports/case_a_room_predictions.csv")
print(df.to_string(index=False))
print("\nDistribusi status:", df["predicted_status"].value_counts().to_dict())

          timestamp device_id  temperature_c  humidity_pct  co2_ppm predicted_status
2026-10-22T08:11:42    room01          27.00         49.95     1487           nyaman
2026-10-22T08:11:42    room01          23.26         66.67      477     tidak_nyaman
2026-10-22T08:11:43    room01          25.38         44.52      580           nyaman
2026-10-22T08:11:43    room01          25.27         49.59     1040           nyaman
2026-10-22T08:11:43    room01          28.62         54.47      985     tidak_nyaman
2026-10-22T08:11:44    room01          23.70         44.26     1283           nyaman
2026-10-22T08:11:44    room01          29.77         44.37      720     tidak_nyaman
2026-10-22T08:11:44    room01          26.15         44.93      509           nyaman

Distribusi status: {'nyaman': 5, 'tidak_nyaman': 3}


## Penutup

Studi kasus A menyatukan seluruh stack: Random Forest mencapai F1 sekitar 1.00 pada
label aturan kenyamanan, dan pipeline MQTT berjalan ujung ke ujung dengan storage CSV
(InfluxDB + Grafana opsional via `--influx`, panel: suhu, kelembaban, CO2, status,
histori status, alert). Model di `models/case_a_room_rf.joblib`.